In [1]:
import json
from pathlib import Path
from typing import List, Dict, Any


def chunk_text(text, chunk_size=500, overlap=50):

    # Сначала разбиваем по абзацам
    paragraphs = text.split('\n\n')

    chunks = []
    current_chunk = ""

    for para in paragraphs:
        # Если абзац пустой, пропускаем
        if not para.strip():
            continue

        # Если абзац длиннее chunk_size, разбиваем его дополнительно
        if len(para) > chunk_size:
            # Если текущий чанк не пуст, сохраняем его
            if current_chunk:
                chunks.append(current_chunk.strip())
                current_chunk = ""

            # Разбиваем длинный абзац на куски
            start = 0
            while start < len(para):
                end = start + chunk_size
                chunk_piece = para[start:end]
                chunks.append(chunk_piece.strip())
                start = end - overlap if overlap > 0 else end
        else:
            # Пытаемся добавить абзац к текущему чанку
            if current_chunk:
                test_chunk = current_chunk + "\n\n" + para
                if len(test_chunk) <= chunk_size:
                    current_chunk = test_chunk
                else:
                    # Текущий чанк заполнен, сохраняем и начинаем новый
                    chunks.append(current_chunk.strip())
                    current_chunk = para
            else:
                current_chunk = para

    # Добавляем последний чанк
    if current_chunk:
        chunks.append(current_chunk.strip())

    # Если чанков нет, возвращаем исходный текст как один чанк
    if not chunks and text:
        chunks = [text[:chunk_size]]

    return chunks

In [2]:
def create_chunks():

    input_path = "documents.jsonl"
    output_path = "chunks.jsonl"


    # Читаем все документы
    documents = []
    with open(input_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                documents.append(json.loads(line))

    print(f"Найдено документов: {len(documents)}")

    total_chunks = 0
    chunk_id_counter = 0

    with open(output_path, 'w', encoding='utf-8') as out_f:
        for doc in documents:
            doc_id = doc.get('doc_id')
            text = doc.get('text', '')
            name = doc.get('name', '')

            # Получаем чанки для документа
            chunks = chunk_text(text)

            for idx, chunk in enumerate(chunks):
                if not chunk.strip():
                    continue

                chunk_id = f"{doc_id}_chunk_{idx}"
                record = {
                    "chunk_id": chunk_id,
                    "doc_id": doc_id,
                    "text": chunk,
                    "chunk_index": idx,
                    "name": name
                }
                out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
                chunk_id_counter += 1
                total_chunks += 1

    print(f"Создано {total_chunks} чанков в {output_path}")
    print(f"В среднем {total_chunks / len(documents):.1f} чанков на документ")

    # Показываем пример
    if total_chunks > 0:
        print("\n📝 Пример первого чанка:")
        with open(output_path, 'r', encoding='utf-8') as f:
            first_line = f.readline()
            example = json.loads(first_line)
            print(f"   chunk_id: {example['chunk_id']}")
            print(f"   doc_id: {example['doc_id']}")
            print(f"   chunk_index: {example['chunk_index']}")
            print(f"   text (первые 150 символов): {example['text'][:150]}...")


create_chunks()

Найдено документов: 5000
Создано 5000 чанков в chunks.jsonl
В среднем 1.0 чанков на документ

📝 Пример первого чанка:
   chunk_id: 0_chunk_0
   doc_id: 0
   chunk_index: 0
   text (первые 150 символов): The payment was deducted from my bank account but the transaction shows failed.

Ответ поддержки: Data synchronization restored after backend service ...
